# Nisjemodell — nasjonal (raster + vektor)

Bygger et nasjonalt NiN/nisjemodell-datasett ved å kombinere nasjonale kilder,
etter samme metodikk som `notebooks/scratch/create_test_dataset_mr.ipynb`, men for
hele Norge og med disse endringene:

1. **Brakkvann** hentes fra det 50 m fylte rasteret `Norfjords_160_surf_salt_mean_UTM33_brakkvann_50m_filled.tif`
   (samme padded 50 m-grid som substrat og fotisk sone).
2. **Substrat** hentes fra det modellerte rasteret
   `nisjedata-substrat-xgbclassifier_norge_latest_25833_2class_padded.tif` (reprojisert med
   WarpedVRT inn på gridet, som brakkvann).
3. **Tørrfall** håndteres i **vektor**-steget (ikke rasterisert på 50 m-gridet), fordi
   tørrfallspolygonene er små og detaljerte og vanskelige å gjengi presist på gridet.
4. **Observasjons-substrat** (`nisjedata-substrat-klassifisering`) legges på som
   **vektor**-overlay etter vektoriseringen, slik at de observerte polygongrensene
   bevares nøyaktig. Lys + salinitet arves fra det underliggende NiN-polygonet, og kun
   substrat-aksen byttes.

Pipeline:
Lys + Salinitet + Substrat → vindus-vis rasterklassifisering (3-bit nøkkel) →
vektorisering → tørrfall-overlay (vektor) →
observasjons-substrat-overlay (vektor) → trekk fra land → lagre.

In [1]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import rasterize as rio_rasterize
from rasterio.vrt import WarpedVRT
from rasterio.windows import Window
from rasterio.enums import Resampling

import mnk
import mnk.sources
import mnk.utils
import mnk.vectorize

CRS = "EPSG:25833"
EPSG = 25833
NODATA = 255

## Kilder og stier

- **Lys / referansegrid**: nasjonalt fotisk-sone-raster (50 m, EPSG:25833, 1=Eufotisk, 0=Afotisk, -1=nodata).
- **Brakkvann**: `hi/Norfjords_160_surf_salt_mean_UTM33_brakkvann.tif` (EPSG:32633, 1=brakk, 0=salt, 255=nodata).
- **Substrat**: nasjonal `nisjedata-substrat_norge` geoparquet (`BunnType`).
- **Tørrfall**: `mnk.sources.sea_map_torrfall()` (alle fylker) — brukes i vektor-steget.
- **Land**: nasjonal `Basisdata_Landareal` geoparquet.

In [ ]:
REPO = Path.cwd()

photic_path   = "gs://niva-geodata/MarintNaturKart/results/nisjedata-fotisk-sone-kd2020-2025_norge_2026_25833_padded.tif"
brakkvann_path = "gs://niva-geodata/MarintNaturKart/results/Norfjords_160_surf_salt_mean_UTM33_brakkvann_50m_filled.tif"

# Modellert substrat-raster (0=løsbunn, 1=blanding, 2=fastbunn, 255=nodata)
substrat_raster_path = "gs://niva-geodata/MarintNaturKart/results/nisjedata-substrat-xgbclassifier_norge_latest_25833_2class_padded.tif"
# Observasjons-substrat (autoritative polygoner, legges på som vektor til slutt)
obs_substrat_parquet = "gs://niva-geodata/MarintNaturKart/results/nisjedata-substrat-klassifisering_norge_latest_25833.geo.parquet"

land_parquet = "gs://niva-geodata/MarintNaturKart/aux/land-mask-norway_FKB-osm-seaborder-subdivided-fullres_v3_25833_20260924.geo.parquet"

nin_raster_path = REPO / "nin_norge_2026_25833.tif"

## NiN-koding

Vi koder tre binære dimensjoner (Lys, Salinitet, Substrat) til en 3-bit nøkkel og
antar foreløpig **Utenfor** tørrfall overalt. Tørrfall legges på i vektor-steget.

```
key = lys*4 + salinitet*2 + substrat
```

In [ ]:
# NiN heltall (1-12) -> (nin_kode_fkid, NiN_name) — samme som MR-notebooken
NIN_INT_MAP = {
     1: ("NA-MC04", "Brakkvanns-sedimentbunn"),          # Eufotisk Brakk Løst  Innenfor
     2: ("NA-MA04", "Fjærebelte-sedimentbunn"),          # Eufotisk Salt  Løst  Innenfor
     3: ("NA-MC01", "Fast brakkvanns-fjærebeltebunn"),   # Eufotisk Brakk Fast  Innenfor
     4: ("NA-MA01", "Fast saltvanns-fjærebeltebunn"),    # Eufotisk Salt  Fast  Innenfor
     5: ("NA-MC04", "Brakkvanns-sedimentbunn"),          # Eufotisk Brakk Løst  Utenfor
     6: ("NA-MC04", "Brakkvanns-sedimentbunn"),          # Afotisk  Brakk Løst  Utenfor
     7: ("NA-MA05", "Eufotisk saltvanns-sedimentbunn"),  # Eufotisk Salt  Løst  Utenfor
     8: ("NA-MA06", "Afotisk saltvanns-sedimentbunn"),   # Afotisk  Salt  Løst  Utenfor
     9: ("NA-MC02", "Fast brakkvannsbunn"),              # Eufotisk Brakk Fast  Utenfor
    10: ("NA-MC02", "Fast brakkvannsbunn"),              # Afotisk  Brakk Fast  Utenfor
    11: ("NA-MA02", "Eufotisk fast saltvannsbunn"),      # Eufotisk Salt  Fast  Utenfor
    12: ("NA-MA03", "Afotisk fast saltvannsbunn"),       # Afotisk  Salt  Fast  Utenfor
}

# 3-bit LUT (tørrfall = Utenfor):  key = lys*4 + salinitet*2 + substrat
#   0: Afotisk Salt  Løst -> 8     1: Afotisk Salt  Fast -> 12
#   2: Afotisk Brakk Løst -> 6     3: Afotisk Brakk Fast -> 10
#   4: Eufot.  Salt  Løst -> 7     5: Eufot.  Salt  Fast -> 11
#   6: Eufot.  Brakk Løst -> 5     7: Eufot.  Brakk Fast -> 9
LUT3 = np.array([8, 12, 6, 10, 7, 11, 5, 9], dtype=np.uint8)

# Tørrfall-remap (Innenfor): kun eufotiske "Utenfor"-koder endres.
# Afotiske koder (6,8,10,12) berøres ikke (tørrfall er intertidalt/eufotisk).
TORRFALL_REMAP = {5: 1, 7: 2, 9: 3, 11: 4}

# Observasjons-substrat-overlay: bytt KUN substrat-aksen på en eksisterende NiN-kode.
# Par (Løst <-> Fast) med samme lys/salinitet/sone:
#   (1,3) (2,4) eufot innenfor;  (5,9) (7,11) eufot utenfor;  (6,10) (8,12) afot.
SUBSTRAT_TO_FAST = {1: 3, 2: 4, 5: 9, 6: 10, 7: 11, 8: 12}
SUBSTRAT_TO_LOST = {v: k for k, v in SUBSTRAT_TO_FAST.items()}
print(LUT3)

## Last observasjons-substrat (nasjonalt)

Substrat for selve gridet hentes fra det modellerte rasteret i loopen under.
Her lastes kun **observasjons**-polygonene (`nisjedata-substrat-klassifisering`),
som legges på som vektor-overlay etter vektoriseringen for å bevare geometrien.

In [ ]:
BUNN_TO_INT = {"løsbunn": 0, "blanding": 0, "fastbunn": 1}  # 0=Løst, 1=Fast

gdf_obs = gpd.read_parquet(obs_substrat_parquet).to_crs(CRS)
gdf_obs = gdf_obs[gdf_obs["BunnType"].isin(BUNN_TO_INT)].reset_index(drop=True)
gdf_obs["_obs_fast"] = gdf_obs["BunnType"].map(BUNN_TO_INT).astype("uint8")
gdf_obs = gdf_obs[["geometry", "_obs_fast"]]
_ = gdf_obs.sindex  # build spatial index
print(f"Observasjons-polygoner: {len(gdf_obs):,}")

## Vindus-vis rasterklassifisering

For hvert rad-stripe-vindu:
1. les Lys fra referansegridet,
2. reprojiser brakkvann (WarpedVRT) inn på gridet,
3. reprojiser modellert substrat-raster (WarpedVRT) inn på gridet
   (2=fastbunn → Fast, ellers Løst),
4. kod til NiN (Utenfor-antakelse) og skriv blokk til midlertidig GeoTIFF.

In [ ]:
BLOCK_ROWS = 2048

with rasterio.open(photic_path) as photic:
    transform = photic.transform
    width, height = photic.width, photic.height
    photic_nodata = photic.nodata if photic.nodata is not None else -1

    profile = dict(
        driver="GTiff", height=height, width=width, count=1, dtype="uint8",
        crs=CRS, transform=transform, nodata=NODATA, compress="deflate",
        tiled=True, blockxsize=512, blockysize=512,
    )

    brak_src = rasterio.open(brakkvann_path)
    brak_vrt = WarpedVRT(
        brak_src, crs=CRS, transform=transform, width=width, height=height,
        resampling=Resampling.nearest, src_nodata=brak_src.nodata, nodata=255,
    )

    sub_src = rasterio.open(substrat_raster_path)
    sub_vrt = WarpedVRT(
        sub_src, crs=CRS, transform=transform, width=width, height=height,
        resampling=Resampling.nearest, src_nodata=sub_src.nodata, nodata=255,
    )

    counts = np.zeros(13, dtype=np.int64)
    with rasterio.open(nin_raster_path, "w", **profile) as dst:
        for row0 in range(0, height, BLOCK_ROWS):
            nrows = min(BLOCK_ROWS, height - row0)
            win = Window(0, row0, width, nrows)
            shp = (nrows, width)

            lys_raw = photic.read(1, window=win)
            lys_valid = lys_raw != photic_nodata
            lys = (lys_raw == 1).astype(np.uint8)  # 1=Eufotisk, 0=Afotisk

            # Salinitet: brakk kun der VRT==1, ellers salt (dekningshull -> salt)
            brak = brak_vrt.read(1, window=win)
            salinitet = (brak == 1).astype(np.uint8)

            # Substrat fra modellert raster: 2=fastbunn -> Fast(1), 0/1 -> Løst(0)
            sub_raw = sub_vrt.read(1, window=win)
            sub_valid = sub_raw != 255
            substrat = (sub_raw == 2).astype(np.uint8)

            valid = lys_valid & sub_valid
            nin = np.full(shp, NODATA, dtype=np.uint8)
            key = (lys[valid] * 4 + salinitet[valid] * 2 + substrat[valid]).astype(np.uint8)
            nin[valid] = LUT3[key]
            dst.write(nin, 1, window=win)

            for v in np.unique(nin[nin != NODATA]):
                counts[v] += int(np.sum(nin == v))
            print(f"  rows {row0:>6}-{row0+nrows:<6} valid={int(valid.sum()):>12,}", end="\r")

    brak_vrt.close()
    brak_src.close()
    sub_vrt.close()
    sub_src.close()

print("\nNiN-fordeling (før tørrfall/observasjon/land):")
for i, (code_, name) in NIN_INT_MAP.items():
    if counts[i]:
        print(f"  {i:2d} {code_} {name}: {counts[i]:,}")

## Vektoriser + generaliser (GRASS)

Vektoriserer NiN-rasteret og fjerner 50 m trappetrinn-artefaktene i samme, disk-streamede operasjon via GRASS `r.to.vect` → `v.generalize` (Douglas–Peucker, terskel 25 m). Topologi bevares slik at felles grenser mellom NiN-klasser ikke får glipper/overlapp. Minnegjerrig fordi GRASS strømmer mot disk.

In [ ]:
nin_vec_path = REPO / "notebooks/data/nin_norge_2026_25833_generalized.gpkg"
mnk.vectorize.with_grass(nin_raster_path, nin_vec_path)

gdf_nin = gpd.read_file(nin_vec_path)
gdf_nin = gdf_nin.rename(columns={"class_int": "nin_int"})
gdf_nin = gdf_nin[gdf_nin["nin_int"] != NODATA].reset_index(drop=True)
gdf_nin = gdf_nin.explode(index_parts=False).reset_index(drop=True)
if gdf_nin.crs is None or gdf_nin.crs.to_epsg() != EPSG:
    gdf_nin = gdf_nin.set_crs(CRS)
gdf_nin[["nin_kode_fkid", "NiN_name"]] = gdf_nin["nin_int"].map(NIN_INT_MAP).apply(pd.Series)
print(f"Polygoner: {len(gdf_nin):,}")
gdf_nin.head()

## Tørrfall i vektor-steget

Tørrfallspolygonene er små og detaljerte. I stedet for å rasterisere dem til 50 m
legger vi dem på som vektor-overlay: NiN-polygoner (eufotiske) som overlapper tørrfall
splittes, og de delene som ligger **innenfor** tørrfall remappes:

`NA-MA05→NA-MA04`, `NA-MC02→NA-MC01`, `NA-MA02→NA-MA01`, `NA-MC04→NA-MC04`.

In [ ]:
gdf_torrfall = mnk.sources.sea_map_torrfall()  # alle fylker
gdf_torrfall = gdf_torrfall.to_crs(CRS)
torrfall_union = gpd.GeoDataFrame(
    geometry=[gdf_torrfall.geometry.union_all()], crs=CRS
)
print(f"Tørrfall-polygoner: {len(gdf_torrfall):,}")

In [ ]:
# Kun koder som faktisk endres av tørrfall trenger å splittes
remap_mask = gdf_nin["nin_int"].isin(TORRFALL_REMAP)
gdf_remap = gdf_nin[remap_mask].reset_index(drop=True)
gdf_keep = gdf_nin[~remap_mask].reset_index(drop=True)

# Innenfor tørrfall: snitt -> remap NiN-kode
inside = gpd.overlay(gdf_remap, torrfall_union, how="intersection", keep_geom_type=True)
inside["nin_int"] = inside["nin_int"].map(TORRFALL_REMAP).astype(inside["nin_int"].dtype)

# Utenfor tørrfall: differanse -> uendret kode
outside = gpd.overlay(gdf_remap, torrfall_union, how="difference", keep_geom_type=True)

gdf_nin_tf = pd.concat([gdf_keep, outside, inside], ignore_index=True)
gdf_nin_tf = gpd.GeoDataFrame(gdf_nin_tf, geometry="geometry", crs=CRS)
gdf_nin_tf = gdf_nin_tf[~gdf_nin_tf.is_empty].explode(index_parts=False).reset_index(drop=True)
# oppdater kode/navn etter remap
gdf_nin_tf[["nin_kode_fkid", "NiN_name"]] = gdf_nin_tf["nin_int"].map(NIN_INT_MAP).apply(pd.Series)
print(f"Polygoner etter tørrfall-overlay: {len(gdf_nin_tf):,}")

## Observasjons-substrat i vektor-steget

Observasjonspolygonene er autoritative for substrat. De legges på som vektor-overlay
slik at de observerte grensene bevares nøyaktig (ingen piksel-snapping):

1. `intersection` med NiN-laget → hver bit arver lys+salinitet fra det underliggende
   polygonet (store observasjoner splittes kun langs lys/salinitet-grenser),
2. kun substrat-aksen byttes via `SUBSTRAT_TO_FAST` / `SUBSTRAT_TO_LOST`,
3. `difference` stanser observasjonsfotavtrykket ut av NiN-laget, og de byttede
   observasjonsbitene limes inn igjen.

In [ ]:
# Observasjons-substrat som vektor-overlay (bevarer polygongeometri)
_dtype = gdf_nin_tf["nin_int"].dtype

# Arv lys+salinitet fra underliggende NiN-polygon, bytt kun substrat-aksen
obs_pieces = gpd.overlay(gdf_obs, gdf_nin_tf, how="intersection", keep_geom_type=True)
_fast = obs_pieces["_obs_fast"] == 1
obs_pieces.loc[_fast, "nin_int"] = (
    obs_pieces.loc[_fast, "nin_int"].map(lambda v: SUBSTRAT_TO_FAST.get(v, v))
)
obs_pieces.loc[~_fast, "nin_int"] = (
    obs_pieces.loc[~_fast, "nin_int"].map(lambda v: SUBSTRAT_TO_LOST.get(v, v))
)
obs_pieces["nin_int"] = obs_pieces["nin_int"].astype(_dtype)

# Stans observasjonsfotavtrykket ut av NiN-laget, lim inn de byttede bitene
nin_minus = gpd.overlay(gdf_nin_tf, gdf_obs[["geometry"]], how="difference", keep_geom_type=True)

gdf_nin_obs = pd.concat([nin_minus, obs_pieces], ignore_index=True)
gdf_nin_obs = gpd.GeoDataFrame(gdf_nin_obs, geometry="geometry", crs=CRS)
gdf_nin_obs = gdf_nin_obs[~gdf_nin_obs.is_empty].explode(index_parts=False).reset_index(drop=True)
gdf_nin_obs[["nin_kode_fkid", "NiN_name"]] = gdf_nin_obs["nin_int"].map(NIN_INT_MAP).apply(pd.Series)
gdf_nin_obs = gdf_nin_obs[["nin_int", "nin_kode_fkid", "NiN_name", "geometry"]]
print(f"Polygoner etter observasjons-overlay: {len(gdf_nin_obs):,}")

## Trekk fra land

## Klipp til grunnlinje (1 nautisk mil)

Fjern polygoner utenfor grunnlinje + 1 nautisk mil, før land trekkes fra.

In [ ]:
print(f"Polygoner før grunnlinje-klipp: {len(gdf_nin_obs):,}")
gdf_nin_obs = mnk.vectorize.clip_to_grunnlinje(gdf_nin_obs)
print(f"Polygoner etter grunnlinje-klipp: {len(gdf_nin_obs):,}")

In [ ]:
land = gpd.read_parquet(land_parquet)
land = land.to_crs(CRS)

gdf_nin_land = mnk.vectorize.subtract_land(gdf_nin_obs, land)
print(f"Polygoner etter land-subtraksjon: {len(gdf_nin_land):,}")

## Lagre resultat

In [ ]:
fname = mnk.utils.to_filename("nisjemodell", "norge", "2026", CRS.split(":")[1])
out_dir = REPO
gdf_nin_land.to_file(out_dir / f"{fname}.gpkg", driver="GPKG", layer="nin")
gdf_nin_land.to_parquet(out_dir / f"{fname}.geo.parquet", compression="snappy")
print(f"Lagret {len(gdf_nin_land):,} polygoner -> {fname}.gpkg / .geo.parquet")

In [ ]:
# Last opp til PostGIS (valgfritt)
mnk.utils.to_postgis(gdf_nin_land, fname)